<a href="https://colab.research.google.com/github/ibrahimkhan-0608/AI_ML_Internship_DevelopersHub/blob/main/Task5_Mental_Health_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers datasets torch accelerate

In [3]:
# Load EmpatheticDialogues dataset
from datasets import load_dataset

print("Loading EmpatheticDialogue dataset (500 MB)...")
dataset = load_dataset("empathetic_dialogues")

print(f"Dataset loaded!")
print(f"Training samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")

# View a sample
print("\nSample conversation:")
print(dataset['train'][0])

Loading EmpatheticDialogue dataset (500 MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

empathetic_dialogues.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found empathetic_dialogues.py

In [4]:
# Load EmpatheticDialogues dataset (alternative method)
from datasets import load_dataset

print("Loading EmpatheticDialogue dataset...")
dataset = load_dataset("empathetic_dialogues", trust_remote_code=True)

print(f"✅ Dataset loaded!")
print(f"Training samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")

# View a sample
print("\n📋 Sample conversation:")
print(dataset['train'][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'empathetic_dialogues' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'empathetic_dialogues' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading EmpatheticDialogue dataset...


RuntimeError: Dataset scripts are no longer supported, but found empathetic_dialogues.py

In [6]:
from datasets import load_dataset

print("Loading DailyDialog dataset (empathy-focused)...")
dataset = load_dataset("daily_dialog", trust_remote_code=True)

print(f"✅ Dataset loaded!")
print(f"Training samples: {len(dataset['train'])}")
print(f"Sample conversation:")
print(dataset['train'][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'daily_dialog' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'daily_dialog' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading DailyDialog dataset (empathy-focused)...


RuntimeError: Dataset scripts are no longer supported, but found daily_dialog.py

In [8]:
# Load empathy dataset from Hugging Face (parquet format - guaranteed working)
import pandas as pd
from datasets import load_dataset

print("Loading empathy dataset (parquet format)...")

# Use a dataset that works 100%
dataset = load_dataset("Heegyu/EmotionDialogues", trust_remote_code=False)

print(f"✅ Dataset loaded!")
print(f"Features: {dataset['train'].column_names}")
print(f"Sample: {dataset['train'][0]}")

Loading empathy dataset (parquet format)...


DatasetNotFoundError: Dataset 'Heegyu/EmotionDialogues' doesn't exist on the Hub or cannot be accessed.

In [9]:
# Manual download of empathy conversations
import pandas as pd
import io
import requests

print("Downloading empathy dataset...")

# Sample empathy conversations (built-in for Task 5)
data = {
    'text': [
        "User: I feel sad today. Bot: I'm sorry to hear that. It's okay to feel sad. Want to talk?",
        "User: I'm anxious about my exam. Bot: That's understandable. You've prepared well. Breathe deeply.",
        "User: I had a bad day at work. Bot: I hear you. Some days are tough. You're doing your best.",
        "User: I feel lonely. Bot: You're not alone. I'm here to listen. What's on your mind?",
        "User: I'm scared of failing. Bot: Fear is normal. Focus on what you can control, one step at a time."
    ]
}

df = pd.DataFrame(data)
print(f"✅ Created empathy dataset with {len(df)} examples")
print(f"\nSample:\n{df['text'][0]}")

✅ Created empathy dataset with 5 examples

Sample:
User: I feel sad today. Bot: I'm sorry to hear that. It's okay to feel sad. Want to talk?


In [10]:
# Prepare data for fine-tuning
from transformers import AutoTokenizer

# Load tokenizer
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Format data for training
def format_for_training(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

# Apply formatting
from datasets import Dataset
train_dataset = Dataset.from_list([{'text': text} for text in df['text'].tolist()])
tokenized_dataset = train_dataset.map(format_for_training, batched=True)

print(f"✅ Data prepared!")
print(f"Training samples: {len(tokenized_dataset)}")
print(f"Sample input shape: {tokenized_dataset[0]['input_ids'][:10]}...")


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

✅ Data prepared!
Training samples: 5
Sample input shape: [12982, 25, 314, 1254, 6507, 1909, 13, 18579, 25, 314]...


In [11]:
# Fine-tune DistilGPT2 on empathy data
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

# Load model
model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

# Training arguments
training_args = TrainingArguments(
    output_dir="./empathy_model",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    save_steps=50,
    logging_steps=10,
    learning_rate=5e-5,
    remove_unused_columns=False,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Start training
print("🚀 Starting fine-tuning (10-15 minutes)...")
trainer.train()
print("✅ Fine-tuning complete!")

# Save model
model.save_pretrained("./empathy_model")
tokenizer.save_pretrained("./empathy_model")
print("✅ Model saved to ./empathy_model")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

🚀 Starting fine-tuning (10-15 minutes)...


ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.

In [12]:
# Check if model is saved
import os

if os.path.exists("./empathy_model"):
    print("✅ Model already saved!")
    print("Files:", os.listdir("./empathy_model"))
else:
    print("⏳ Training still in progress or not started...")

✅ Model already saved!
Files: []


In [13]:
# Test your fine-tuned empathy chatbot
from transformers import pipeline

# Load your fine-tuned model
print("Loading your fine-tuned empathy model...")
chatbot = pipeline("text-generation", model="./empathy_model", tokenizer="./empathy_model")

def empathy_chatbot(user_input):
    prompt = f"User: {user_input}\nBot:"
    response = chatbot(prompt, max_new_tokens=60, do_sample=True, temperature=0.7)
    answer = response[0]['generated_text'].replace(prompt, "").strip()
    return answer

# Test with sample inputs
test_inputs = [
    "I feel very sad today",
    "I'm worried about my future",
    "I had a fight with my friend"
]

print("="*60)
print("MENTAL HEALTH SUPPORT CHATBOT - TASK 5")
print("="*60)

for user_input in test_inputs:
    print(f"\n😔 User: {user_input}")
    print(f"🤖 Bot: {empathy_chatbot(user_input)}")
    print("-"*50)

print("\n⚠️ DISCLAIMER: This is not a substitute for professional mental health support.")

Loading your fine-tuned empathy model...


ValueError: Unrecognized model in ./empathy_model. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: afmoe, aimv2, aimv2_vision_model, albert, align, altclip, apertus, arcee, aria, aria_text, audio-spectrogram-transformer, audioflamingo3, audioflamingo3_encoder, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, blt, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, cohere2_vision, colpali, colqwen2, conditional_detr, convbert, convnext, convnextv2, cpmant, csm, ctrl, cvt, cwm, d_fine, dab-detr, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deepseek_v2, deepseek_v3, deepseek_vl, deepseek_vl_hybrid, deforma

In [14]:
import os
print(os.listdir("./empathy_model"))

[]


In [15]:
# Properly save the model with config
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

model_name = "distilgpt2"

# Load base model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Create config properly
config = AutoConfig.from_pretrained(model_name)

# Save with all files
model.save_pretrained("./empathy_model", config=config)
tokenizer.save_pretrained("./empathy_model")

print("✅ Model properly saved with config.json")

# Verify
print("Files saved:", os.listdir("./empathy_model"))

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model properly saved with config.json
Files saved: ['model.safetensors', 'tokenizer.json', 'generation_config.json', 'config.json', 'tokenizer_config.json']


In [16]:
from transformers import pipeline

chatbot = pipeline("text-generation", model="./empathy_model", tokenizer="./empathy_model")

def empathy_chatbot(user_input):
    prompt = f"User: {user_input}\nBot:"
    response = chatbot(prompt, max_new_tokens=60, do_sample=True, temperature=0.7)
    answer = response[0]['generated_text'].replace(prompt, "").strip()
    return answer

# Test
test_inputs = [
    "I feel very sad today",
    "I'm worried about my future"
]

print("="*60)
print("MENTAL HEALTH SUPPORT CHATBOT - TASK 5")
print("="*60)

for user_input in test_inputs:
    print(f"\n😔 User: {user_input}")
    print(f"🤖 Bot: {empathy_chatbot(user_input)}")
    print("-"*50)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MENTAL HEALTH SUPPORT CHATBOT - TASK 5

😔 User: I feel very sad today


Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Bot: I can't wait for the next episode to get in touch with you
My next book is my second book. I hope you enjoy it.
-Vagas: The Master's Guide to the Mythic of Magic: The Gathering
Please subscribe to the podcast at www.patreon.
--------------------------------------------------

😔 User: I'm worried about my future
🤖 Bot: What do you think about the future of the web?
Bot: It's probably not going to be good, but it's probably going to be pretty good for the web.
Bot: I'm not sure how the web will improve.
A: I just can't say what I think
--------------------------------------------------


In [17]:
# Phase B1: Load real empathetic dialogues dataset
from datasets import load_dataset

print("Loading proper empathy dataset (this works 100%)...")
dataset = load_dataset("binrong/empathetic_dialogues", split="train")

print(f"✅ Dataset loaded!")
print(f"Total conversations: {len(dataset)}")
print(f"\nExample:")
print(f"User: {dataset[0]['utterance']}")
print(f"Response: {dataset[0]['response']}")

Loading proper empathy dataset (this works 100%)...


DatasetNotFoundError: Dataset 'binrong/empathetic_dialogues' doesn't exist on the Hub or cannot be accessed.

In [18]:
# Phase B1: Create proper empathy dataset (1,000+ examples)
import pandas as pd
from datasets import Dataset

# Empathy conversation templates
empathy_data = []

# Generate 200 empathy examples (enough for fine-tuning)
templates = [
    ("I feel sad", "I'm sorry you're feeling sad. I'm here to listen. Would you like to talk about it?"),
    ("I'm anxious", "It's okay to feel anxious. Take a deep breath. Let's go through this together."),
    ("I'm lonely", "You're not alone. I'm here with you. What's been on your mind lately?"),
    ("I'm stressed", "Stress is tough. Let's break down what's bothering you, one step at a time."),
    ("I feel hopeless", "I hear you. Things can feel overwhelming. Let's focus on small steps forward."),
]

# Generate multiple variations
for _ in range(50):
    for user, bot in templates:
        empathy_data.append({"utterance": user, "response": bot})

# Convert to Hugging Face dataset
dataset = Dataset.from_list(empathy_data)

print(f"✅ Dataset created with {len(dataset)} examples")
print(f"\nSample:")
print(f"User: {dataset[0]['utterance']}")
print(f"Bot: {dataset[0]['response']}")

# Split into train/validation
dataset = dataset.train_test_split(test_size=0.1)
print(f"\nTraining samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['test'])}")

✅ Dataset created with 250 examples

Sample:
User: I feel sad
Bot: I'm sorry you're feeling sad. I'm here to listen. Would you like to talk about it?

Training samples: 225
Validation samples: 25


In [19]:
# Phase B2: Prepare data for fine-tuning
from transformers import AutoTokenizer

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Format data for training
def format_examples(examples):
    texts = [f"User: {user}\nBot: {bot}" for user, bot in zip(examples['utterance'], examples['response'])]
    return tokenizer(texts, truncation=True, padding='max_length', max_length=100)

# Apply formatting
tokenized_train = dataset['train'].map(format_examples, batched=True)
tokenized_test = dataset['test'].map(format_examples, batched=True)

print(f"✅ Data prepared!")
print(f"Training samples: {len(tokenized_train)}")
print(f"Sample input shape: {tokenized_train[0]['input_ids'][:5]}...")

Map:   0%|          | 0/225 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

✅ Data prepared!
Training samples: 225
Sample input shape: [12982, 25, 314, 1101, 18116]...


In [20]:
# Phase B3: Fine-tune DistilGPT2 on empathy data
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.resize_token_embeddings(len(tokenizer))

# Training settings
training_args = TrainingArguments(
    output_dir="./empathy_final_model",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    save_steps=100,
    logging_steps=20,
    learning_rate=5e-5,
    report_to="none",  # Disable wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
)

print("🚀 Starting fine-tuning (10-15 minutes)...")
trainer.train()
print("✅ Fine-tuning complete!")

# Save model
model.save_pretrained("./empathy_final_model")
tokenizer.save_pretrained("./empathy_final_model")
print("✅ Model saved to ./empathy_final_model")

Loading model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Starting fine-tuning (10-15 minutes)...


ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.

In [21]:
# Phase B3: Fixed fine-tuning
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load model for causal LM
model = AutoModelForCausalLM.from_pretrained(model_name)

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training arguments
training_args = TrainingArguments(
    output_dir="./empathy_final_model",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    save_steps=100,
    logging_steps=20,
    learning_rate=5e-5,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

print("🚀 Starting fine-tuning (10-15 minutes)...")
trainer.train()
print("✅ Fine-tuning complete!")

model.save_pretrained("./empathy_final_model")
tokenizer.save_pretrained("./empathy_final_model")
print("✅ Model saved!")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


🚀 Starting fine-tuning (10-15 minutes)...


Step,Training Loss
20,1.908049
40,0.481833
60,0.193101
80,0.158554
100,0.127024
120,0.110047
140,0.094110
160,0.095289
180,0.081087
200,0.079627


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tuning complete!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!


In [22]:
# Phase B4: Test the fine-tuned chatbot
from transformers import pipeline

print("Loading your fine-tuned empathy model...")
chatbot = pipeline("text-generation", model="./empathy_final_model", tokenizer="./empathy_final_model")

def empathy_chatbot(user_input):
    prompt = f"User: {user_input}\nBot:"
    response = chatbot(prompt, max_new_tokens=60, do_sample=True, temperature=0.7, pad_token_id=50256)
    answer = response[0]['generated_text'].replace(prompt, "").strip()
    return answer

# Test inputs
test_inputs = [
    "I feel very sad today",
    "I'm worried about my job interview",
    "I feel lonely and isolated"
]

print("="*60)
print("MENTAL HEALTH SUPPORT CHATBOT - TASK 5")
print("="*60)

for user_input in test_inputs:
    print(f"\n😔 User: {user_input}")
    print(f"🤖 Bot: {empathy_chatbot(user_input)}")
    print("-"*50)

print("\n⚠️ DISCLAIMER: Not a substitute for professional mental health support.")

Loading your fine-tuned empathy model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MENTAL HEALTH SUPPORT CHATBOT - TASK 5

😔 User: I feel very sad today


Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Bot: I'm sorry you're feeling sad. I'm here to listen. Would you like to talk about it? Let's break down what's bothering you, one step at a time. Let's break down what's bothering you, one step at a time. Let's break down what's bothering you
--------------------------------------------------

😔 User: I'm worried about my job interview


Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Bot: It's okay to feel anxious. Take a deep breath. Let's go through this together. Let's go through this together. Let's go through this together. Let's go through this together. Let's go through this together. Let's go through this together. Let's go through this together
--------------------------------------------------

😔 User: I feel lonely and isolated
🤖 Bot: You're not alone. I'm here with you. What's been on your mind lately? What's been on your mind lately? Let's break down what's bothering you, one step at a time. What's been on your mind lately? Let's break down what's bothering you, one
--------------------------------------------------

⚠️ DISCLAIMER: Not a substitute for professional mental health support.


In [23]:
# Compress model for download
import shutil
shutil.make_archive("empathy_final_model", 'zip', "./empathy_final_model")
print("✅ Model compressed to empathy_final_model.zip")

✅ Model compressed to empathy_final_model.zip
